# General Norms

Showing overrepresented nodes at a coarse-grained network model.

## 0. Set Up

In [1]:
import numpy as np

import pandas as pd

from pathlib import Path

from network_pluralism.access.resolved import to_level
from network_pluralism.access.projected import to_projection
from network_pluralism.access.utils.labels import (
    get_well_written_sections,
)

from sklearn.preprocessing import normalize

In [2]:
EXPORT_PATH = Path("../../figures_and_results/general_norms")
EXPORT_PATH.mkdir(parents=True, exist_ok=True)

## 1. Computation

In [3]:
def get_ranks(
    matrix, c, minimum_coreference=10, threshold=1 / 4, top_n=None, well_written=False
):
    p = matrix.copy()
    p.data[p.data <= minimum_coreference] = 0
    p.eliminate_zeros()

    number_of_sections = (p.sum(axis=1) > 1).astype(int).sum()
    m_max_norm = normalize(p, "max", axis=1)

    B = (m_max_norm > threshold).astype(int)
    rank = pd.DataFrame(
        B.sum(axis=0).A1, index=list(c.keys()), columns=["influence"]
    ).sort_values("influence", ascending=False)
    if top_n:
        rank = rank.head(10)
    rank["share"] = rank["influence"] / number_of_sections
    rank["Percentage"] = (rank["share"] * 100).round(2)
    if well_written:
        rank["Section"] = get_well_written_sections(rank.index)
        rank.reset_index().set_index("Section")

    return rank

In [4]:
m_dec, r_dec, c_dec = to_level(2, 1)  # Decision Level
m_para, r_para, c_para = to_level(3, 1)  # Paragraph Level


Out Matrix of shape (74214, 93645)


Out Matrix of shape (3111483, 93645)


In [5]:
p_dec = to_projection(m_dec)
p_dec.setdiag(0)
p_dec.eliminate_zeros()

p_para = to_projection(m_para)
p_para.setdiag(0)
p_para.eliminate_zeros()

In [6]:
ranks_dec = get_ranks(p_dec, c_dec, minimum_coreference=20, threshold=1 / 4, top_n=15)
ranks_dec

,influence,share,Percentage
vwgo_154,486,0.178545,17.85
zpo_563,388,0.142542,14.25
fgo_126,372,0.136664,13.67
gg_3,345,0.126745,12.67
zpo_97,318,0.116826,11.68
fgo_135,282,0.103600,10.36
zpo_562,264,0.096988,9.70
vwgo_137,253,0.092946,9.29
gkg_2004_52,246,0.090375,9.04
gg_2,223,0.081925,8.19


In [7]:
ranks_para = get_ranks(p_para, c_para, minimum_coreference=20, threshold=1 / 4, top_n=15)
ranks_para

,influence,share,Percentage
gg_2,53,0.019350,1.94
gg_103,47,0.017160,1.72
gg_3,44,0.016064,1.61
bgb_823,32,0.011683,1.17
gg_19,32,0.011683,1.17
sgg_202,29,0.010588,1.06
gg_20,29,0.010588,1.06
vwgo_132,28,0.010223,1.02
bgb_280,27,0.009858,0.99
zpo_574,26,0.009493,0.95


In [8]:
combined_df = pd.merge(
    ranks_dec.head(10),
    ranks_dec.loc[ranks_dec.head(10).index],
    left_index=True,
    right_index=True,
)[["share_x", "share_y"]]
combined_df["share_x"] = np.round(combined_df["share_x"] * 100, 2)
combined_df["share_y"] = np.round(combined_df["share_y"] * 100, 2)

combined_df["Section"] = get_well_written_sections(combined_df.index)
combined_df.set_index("Section", inplace=True)
combined_df.rename({"share_x": "D", "share_y": "P"}, inplace=True, axis=1)
combined_df.to_csv(EXPORT_PATH / "combined_rank.csv")
combined_df

,D,P
Section,,
§ 154 VwGO,17.85,17.85
§ 563 ZPO,14.25,14.25
§ 126 FGO,13.67,13.67
Art. 3 GG,12.67,12.67
§ 97 ZPO,11.68,11.68
§ 135 FGO,10.36,10.36
§ 562 ZPO,9.70,9.70
§ 137 VwGO,9.29,9.29
§ 52 GKG 2004,9.04,9.04


In [9]:
combined_df.columns = pd.MultiIndex.from_tuples(
    [
        (r"\textbf{Share}", r"\textbf{D}"),
        (r"\textbf{Share}", r"\textbf{P}"),
    ]
)
combined_df.index.name = r"\textbf{Section}"

combined_df.to_latex(
    EXPORT_PATH / "combined_rank.tex",
    escape=False,
    column_format="lcc",  # index + 2 columns
    multicolumn=True,
    multicolumn_format="c",
    float_format="%.2f",
)